# Objective 1B: U-Net DEM-Based Fault Detection

**Model**: U-Net with ResNet34 encoder (ImageNet pretrained)  
**Input**: 3-channel DEM (hillshade, slope, hillshade duplicated)  
**Output**: Binary segmentation mask (fault / no-fault)  
**Study Area**: Parkfield, CA (lat 35.27, lon -119.82)  
**Runtime**: Google Colab A100 GPU  

## Results
| Metric | Value |
|--------|-------|
| mIoU | 0.6729 |
| IoU_fault | 0.3554 |
| F1 | 0.5244 |
| Accuracy | 0.9904 |

In [1]:
# Cell 1: Install packages
!pip install -q segmentation-models-pytorch albumentations
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 16.5 MB/s eta 0:00:00
Done


In [2]:
# Cell 2: Mount Google Drive + verify data
from google.colab import drive
drive.mount('/content/drive')

import os, torch
import numpy as np
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/prithvi_fault')
PATCH_DIR   = PROJECT_DIR / 'data' / 'patches' / 'parkfield_dem'
CKPT_DIR    = PROJECT_DIR / 'checkpoints_unet'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print('\nDataset summary:')
for split in ['train', 'val', 'test']:
    d  = np.load(PATCH_DIR / f'{split}.npz')
    nf = (d['masks'].sum(axis=(1,2)) > 0).sum()
    fp = d['masks'].mean() * 100
    print(f'  {split}: {len(d["images"])} patches | '
          f'fault patches={nf} | fault pixels={fp:.2f}%')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB

Dataset summary:
  train: 2733 patches | fault patches=159 | fault pixels=1.20%
  val: 585 patches | fault patches=34 | fault pixels=1.39%
  test: 588 patches | fault patches=35 | fault pixels=1.15%


In [3]:
# Cell 3: Dataset + DataLoader
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class DEMFaultDataset(Dataset):
    """
    Dataset for DEM-based fault detection.
    Input:  2-channel DEM (hillshade + slope), shape (N, 2, H, W)
    Output: binary fault mask, shape (N, H, W)
    Converts to 3-channel [hillshade, slope, hillshade]
    to match ImageNet pretrained ResNet34 input format.
    """
    def __init__(self, npz_path, augment=False):
        data  = np.load(npz_path)
        imgs  = data['images'].astype(np.float32)  # (N, 2, H, W)
        masks = data['masks'].astype(np.int64)      # (N, H, W)
        imgs  = np.nan_to_num(imgs, nan=0.0)

        # Per-channel min-max normalization to [0, 1]
        for c in range(imgs.shape[1]):
            mn, mx = imgs[:,c].min(), imgs[:,c].max()
            if mx > mn:
                imgs[:,c] = (imgs[:,c] - mn) / (mx - mn)

        # 2-channel -> 3-channel: [hillshade, slope, hillshade]
        imgs_3ch = np.stack([imgs[:,0], imgs[:,1], imgs[:,0]], axis=1)

        self.images    = torch.from_numpy(imgs_3ch).float()
        self.masks     = torch.from_numpy(masks)
        self.augment   = augment
        self.has_fault = (masks.sum(axis=(1,2)) > 0)

        n, nf = len(self.images), int(self.has_fault.sum())
        print(f'  {Path(npz_path).name}: {n} patches, fault={nf} ({nf/n*100:.0f}%)')

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx].clone()   # (3, H, W)
        mask = self.masks[idx].clone()    # (H, W)

        if self.augment:
            if torch.rand(1) > 0.5:
                img  = torch.flip(img,  [-1])
                mask = torch.flip(mask, [-1])
            if torch.rand(1) > 0.5:
                img  = torch.flip(img,  [-2])
                mask = torch.flip(mask, [-2])
            if torch.rand(1) > 0.5:
                k    = torch.randint(1, 4, (1,)).item()
                img  = torch.rot90(img,  k, [-2,-1])
                mask = torch.rot90(mask, k, [-2,-1])
            # Brightness jitter for DEM augmentation
            if torch.rand(1) > 0.5:
                img = img * (0.8 + torch.rand(1) * 0.4)
                img = img.clamp(0, 1)

        return img, mask


print('Loading datasets...')
train_ds = DEMFaultDataset(PATCH_DIR / 'train.npz', augment=True)
val_ds   = DEMFaultDataset(PATCH_DIR / 'val.npz',   augment=False)
test_ds  = DEMFaultDataset(PATCH_DIR / 'test.npz',  augment=False)

# Oversample fault patches (15x) to handle class imbalance
weights = np.where(train_ds.has_fault, 15.0, 1.0)
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(weights).float(),
    num_samples=len(weights),
    replacement=True
)

BATCH_SIZE    = 32
LOADER_KWARGS = dict(num_workers=4, pin_memory=True, persistent_workers=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **LOADER_KWARGS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   **LOADER_KWARGS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,   **LOADER_KWARGS)

imgs, masks = next(iter(train_loader))
fault_ratio = (masks.sum(dim=(1,2)) > 0).float().mean()
print(f'Batch shape: {imgs.shape}')
print(f'Fault patch ratio in batch: {fault_ratio:.2f}')
print('DataLoader ready')

Loading datasets...
  train.npz: 2733 patches, fault=159 (6%)
  val.npz: 585 patches, fault=34 (6%)
  test.npz: 588 patches, fault=35 (6%)
Batch shape: torch.Size([32, 3, 128, 128])
Fault patch ratio in batch: 0.56
DataLoader ready


In [4]:
# Cell 4: Model + Loss + Optimizer
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# U-Net with ResNet34 encoder (ImageNet pretrained)
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=2,
    activation=None
).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}')

# Sanity check
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 128, 128).to(device)
    out   = model(dummy)
print(f'Forward OK: {dummy.shape} -> {out.shape}')


class FaultLoss(nn.Module):
    """
    Combined loss: Weighted CrossEntropy + Dice Loss
    - fault_weight=5 to handle fault pixel imbalance (~5%)
    - Dice loss directly optimizes overlap between prediction and ground truth
    """
    def __init__(self, fault_weight=5.0):
        super().__init__()
        w       = torch.tensor([1.0, fault_weight], dtype=torch.float32).to(device)
        self.ce = nn.CrossEntropyLoss(weight=w)

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        prob    = torch.softmax(inputs, dim=1)[:, 1]
        tgt_f   = (targets == 1).float()
        inter   = (prob * tgt_f).sum(dim=(1,2))
        union   = prob.sum(dim=(1,2)) + tgt_f.sum(dim=(1,2))
        dice    = 1.0 - (2*inter + 1) / (union + 1)
        return ce_loss + dice.mean()


def compute_metrics(logits, targets):
    preds = logits.argmax(1)
    tp = ((preds==1)&(targets==1)).sum().float()
    fp = ((preds==1)&(targets==0)).sum().float()
    fn = ((preds==0)&(targets==1)).sum().float()
    tn = ((preds==0)&(targets==0)).sum().float()
    return {
        'mIoU':      ((tp/(tp+fp+fn+1e-8) + tn/(tn+fp+fn+1e-8)) / 2).item(),
        'IoU_fault':  (tp/(tp+fp+fn+1e-8)).item(),
        'F1':         (2*tp/(2*tp+fp+fn+1e-8)).item(),
        'Acc':        ((tp+tn)/(tp+tn+fp+fn+1e-8)).item(),
    }


NUM_EPOCHS = 100
PATIENCE   = 20

criterion = FaultLoss(fault_weight=5.0)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda')

print('Config:')
print(f'  Loss: CrossEntropy(weight=5) + Dice')
print(f'  LR: 3e-4 -> CosineAnnealing')
print(f'  Epochs: {NUM_EPOCHS}, Patience: {PATIENCE}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Total params:     24,436,514
Trainable params: 24,436,514
Forward OK: torch.Size([2, 3, 128, 128]) -> torch.Size([2, 2, 128, 128])
Config:
  Loss: CrossEntropy(weight=5) + Dice
  LR: 3e-4 -> CosineAnnealing
  Epochs: 100, Patience: 20


In [5]:
# Cell 5: Training loop
import time, json

best_miou    = 0.0
patience_cnt = 0
history      = {'epoch':[], 'train_loss':[], 'val_miou':[],
                'val_iou_fault':[], 'val_f1':[]}

print('='*60)
print('U-Net DEM Fault Detection')
print('='*60)
t0 = time.time()

for epoch in range(1, NUM_EPOCHS+1):

    # Train
    model.train()
    t_loss = 0.0
    for imgs, masks in train_loader:
        imgs  = imgs.float().to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            out  = model(imgs)
            loss = criterion(out, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
    t_loss /= len(train_loader)
    scheduler.step()

    # Validate
    model.eval()
    vm = {'mIoU': 0.0, 'IoU_fault': 0.0, 'F1': 0.0, 'Acc': 0.0}
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.float().to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
            m = compute_metrics(out, masks)
            for k in vm: vm[k] += m[k]
    for k in vm: vm[k] /= len(val_loader)

    history['epoch'].append(epoch)
    history['train_loss'].append(t_loss)
    history['val_miou'].append(vm['mIoU'])
    history['val_iou_fault'].append(vm['IoU_fault'])
    history['val_f1'].append(vm['F1'])

    if epoch % 5 == 0 or epoch <= 3:
        elapsed = (time.time()-t0)/60
        print(f'Ep {epoch:3d} | Loss:{t_loss:.4f} | '
              f'mIoU:{vm["mIoU"]:.4f} | '
              f'IoU_fault:{vm["IoU_fault"]:.4f} | '
              f'F1:{vm["F1"]:.4f} | {elapsed:.1f}min')

    # Save best checkpoint (monitored by IoU_fault)
    if vm['IoU_fault'] > best_miou:
        best_miou    = vm['IoU_fault']
        patience_cnt = 0
        torch.save({'epoch': epoch,
                    'model_state': model.state_dict(),
                    'val_metrics': vm},
                   CKPT_DIR / 'best_unet.pth')
        print(f'  Saved best checkpoint (epoch={epoch}, IoU_fault={best_miou:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    if epoch % 10 == 0:
        with open(CKPT_DIR / 'history.json', 'w') as f:
            json.dump(history, f)

print(f'\nDone in {(time.time()-t0)/60:.1f} min | Best IoU_fault: {best_miou:.4f}')

U-Net DEM Fault Detection
Ep   1 | Loss:1.4036 | mIoU:0.4421 | IoU_fault:0.0250 | F1:0.0397 | 0.1min
  Saved best checkpoint (epoch=1, IoU_fault=0.0250)
Ep   2 | Loss:1.2481 | mIoU:0.4722 | IoU_fault:0.0215 | F1:0.0347 | 0.2min
Ep   3 | Loss:1.0728 | mIoU:0.4814 | IoU_fault:0.0351 | F1:0.0525 | 0.3min
  Saved best checkpoint (epoch=3, IoU_fault=0.0351)
  Saved best checkpoint (epoch=4, IoU_fault=0.0592)
Ep   5 | Loss:0.7426 | mIoU:0.4847 | IoU_fault:0.0227 | F1:0.0372 | 0.4min
Ep  10 | Loss:0.5363 | mIoU:0.5059 | IoU_fault:0.0356 | F1:0.0531 | 0.7min
Ep  15 | Loss:0.4359 | mIoU:0.5156 | IoU_fault:0.0544 | F1:0.0710 | 0.9min
Ep  20 | Loss:0.3459 | mIoU:0.5229 | IoU_fault:0.0615 | F1:0.0732 | 1.2min
  Saved best checkpoint (epoch=20, IoU_fault=0.0615)
  Saved best checkpoint (epoch=23, IoU_fault=0.0628)
Ep  25 | Loss:0.2832 | mIoU:0.5244 | IoU_fault:0.0600 | F1:0.0745 | 1.4min
  Saved best checkpoint (epoch=28, IoU_fault=0.0639)
Ep  30 | Loss:0.2135 | mIoU:0.5256 | IoU_fault:0.0627 | F1:

In [6]:
# Cell 6: Test evaluation + threshold tuning
import numpy as np

ckpt = torch.load(CKPT_DIR / 'best_unet.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded checkpoint: epoch={ckpt["epoch"]}, '
      f'val_IoU_fault={ckpt["val_metrics"]["IoU_fault"]:.4f}')

# Threshold tuning on validation set
val_loader_eval = DataLoader(val_ds, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=0)
model.eval()
all_probs, all_masks_v = [], []
with torch.no_grad():
    for imgs, masks in val_loader_eval:
        imgs = imgs.float().to(device)
        with torch.amp.autocast('cuda'):
            out = model(imgs)
        all_probs.append(out.softmax(1)[:,1].cpu().numpy())
        all_masks_v.append(masks.numpy())

all_probs   = np.concatenate(all_probs).ravel()
all_masks_v = np.concatenate(all_masks_v).ravel()

print('\nThreshold  IoU_fault   F1')
print('-'*35)
best_t, best_iou = 0.5, 0.0
for t in np.arange(0.05, 0.55, 0.05):
    pred = (all_probs > t).astype(int)
    tp   = ((pred==1)&(all_masks_v==1)).sum()
    fp   = ((pred==1)&(all_masks_v==0)).sum()
    fn   = ((pred==0)&(all_masks_v==1)).sum()
    iou  = tp / (tp+fp+fn+1e-8)
    f1   = 2*tp / (2*tp+fp+fn+1e-8)
    print(f'  {t:.2f}      {iou:.4f}     {f1:.4f}')
    if iou > best_iou:
        best_iou, best_t = iou, t
print(f'\nOptimal threshold: {best_t:.2f} (val IoU_fault={best_iou:.4f})')

# Final test evaluation
THRESH = best_t
test_loader_eval = DataLoader(test_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)
all_probs_t, all_masks_t = [], []
with torch.no_grad():
    for imgs, masks in test_loader_eval:
        imgs = imgs.float().to(device)
        with torch.amp.autocast('cuda'):
            out = model(imgs)
        all_probs_t.append(out.softmax(1)[:,1].cpu().numpy())
        all_masks_t.append(masks.numpy())

all_probs_t = np.concatenate(all_probs_t).ravel()
all_masks_t = np.concatenate(all_masks_t).ravel()

pred_t = (all_probs_t > THRESH).astype(int)
tp = ((pred_t==1)&(all_masks_t==1)).sum()
fp = ((pred_t==1)&(all_masks_t==0)).sum()
fn = ((pred_t==0)&(all_masks_t==1)).sum()
tn = ((pred_t==0)&(all_masks_t==0)).sum()

iou_fault = tp / (tp+fp+fn+1e-8)
iou_bg    = tn / (tn+fp+fn+1e-8)
miou      = (iou_fault + iou_bg) / 2
f1        = 2*tp / (2*tp+fp+fn+1e-8)
acc       = (tp+tn) / (tp+tn+fp+fn+1e-8)

print(f'\n{"="*50}')
print(f'TEST RESULTS (threshold={THRESH:.2f})')
print('='*50)
print(f'  mIoU      : {miou:.4f}')
print(f'  IoU_fault : {iou_fault:.4f}')
print(f'  F1        : {f1:.4f}')
print(f'  Accuracy  : {acc:.4f}')

import json
with open(CKPT_DIR / 'test_results.json', 'w') as f:
    json.dump({'threshold': float(THRESH),
               'mIoU': float(miou), 'IoU_fault': float(iou_fault),
               'F1': float(f1), 'Accuracy': float(acc)}, f, indent=2)
print('Saved: checkpoints_unet/test_results.json')

Loaded checkpoint: epoch=28, val_IoU_fault=0.0639

Threshold  IoU_fault   F1
-----------------------------------
  0.05      0.3320     0.4985
  0.10      0.3382     0.5055
  0.15      0.3409     0.5084
  0.20      0.3426     0.5103
  0.25      0.3440     0.5119
  0.30      0.3450     0.5130
  0.35      0.3456     0.5137
  0.40      0.3463     0.5145
  0.45      0.3466     0.5148
  0.50      0.3469     0.5151

Optimal threshold: 0.50 (val IoU_fault=0.3469)

TEST RESULTS (threshold=0.50)
  mIoU      : 0.6729
  IoU_fault : 0.3554
  F1        : 0.5244
  Accuracy  : 0.9904
Saved: checkpoints_unet/test_results.json


In [7]:
# Cell 7: Visualization
import matplotlib.pyplot as plt
import numpy as np

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], 'b')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(alpha=0.3)
axes[1].plot(history['val_miou'],      'g-',  label='mIoU')
axes[1].plot(history['val_iou_fault'], 'r--', label='IoU_fault')
axes[1].plot(history['val_f1'],        'b:',  label='F1')
axes[1].axhline(best_miou, color='r', ls=':', alpha=0.5,
                label=f'Best IoU_fault:{best_miou:.4f}')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'unet_training_curve.png', dpi=150)
plt.show()

# Test prediction visualization
data      = np.load(PATCH_DIR / 'test.npz')
imgs_raw  = data['images']   # (N, 2, H, W)
masks_all = data['masks']
fault_idx = np.where(masks_all.sum(axis=(1,2)) > 0)[0][:6]
n         = len(fault_idx)

fig, axes = plt.subplots(n, 4, figsize=(16, n*4))
if n == 1: axes = axes[np.newaxis]

model.eval()
with torch.no_grad():
    for i, idx in enumerate(fault_idx):
        img_raw  = imgs_raw[idx].astype(np.float32)
        img_norm = np.zeros_like(img_raw)
        for c in range(2):
            mn, mx = imgs_raw[:,c].min(), imgs_raw[:,c].max()
            img_norm[c] = np.clip((img_raw[c] - mn) / (mx - mn + 1e-8), 0, 1)
        img_3ch = np.stack([img_norm[0], img_norm[1], img_norm[0]])
        img_t   = torch.from_numpy(img_3ch).float().unsqueeze(0).to(device)

        with torch.amp.autocast('cuda'):
            prob = model(img_t).softmax(1)[0,1].cpu().numpy()

        hs = img_norm[0]
        gt = masks_all[idx]

        axes[i,0].imshow(hs, cmap='gray')
        axes[i,0].set_title('Hillshade'); axes[i,0].axis('off')
        axes[i,1].imshow(gt, cmap='Reds', vmin=0, vmax=1)
        axes[i,1].set_title('Ground Truth'); axes[i,1].axis('off')
        axes[i,2].imshow(prob, cmap='Reds', vmin=0, vmax=1)
        axes[i,2].set_title('Pred (prob)'); axes[i,2].axis('off')
        axes[i,3].imshow(hs, cmap='gray')
        axes[i,3].imshow(prob > THRESH, cmap='Reds', alpha=0.6, vmin=0, vmax=1)
        axes[i,3].set_title(f'Overlay (t={THRESH:.2f})'); axes[i,3].axis('off')

plt.suptitle('U-Net DEM Fault Detection — Test Predictions', fontsize=13)
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'unet_test_predictions.png', dpi=150)
plt.show()
print('Done!')

Output hidden; open in https://colab.research.google.com to view.